# Chapter 15 &mdash; Undecidable Problems Are "$A_{TM}$ in Disguise"

**Concept 9 of the Chapter 15 decomposition:** *Undecidable Problems Are "$A_{TM}$ in Disguise"*

Every undecidability proof in the chapter traces back to $A_{TM}$ through a chain of reductions.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter15/Concept-A-TM-In-Disguise/Concept-A-TM-In-Disguise.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Step back and the chapter has one theorem and many corollaries.

$A_{TM}$ is undecidable by diagonalization (Concept 5). Everything else is reached by
a **chain of mapping reductions**:

```
A_TM  ->  Halt_TM
      ->  complement(E_TM)
      ->  PCP  ->  CFG ambiguity
                -> CFG intersection emptiness
                -> predicate-logic validity
                -> tiling the plane
```

So when you meet a new problem and suspect it is undecidable, the question is not
"how do I diagonalize?" but **"which of these is it in disguise?"**

**Rice's theorem** is the general statement for the first branch: *every*
non-trivial property of the **language** of a TM is undecidable. Most of the standard
examples are instances of it.

## 2. Definitions

### The reduction graph

In [ ]:
EDGES = [("A_TM", "Halt_TM",                  "M' halts iff M accepts"),
         ("A_TM", "complement(E_TM)",         "M_w ignores input, runs M on w"),
         ("A_TM", "PCP",                      "computation-history tiles"),
         ("PCP",  "CFG ambiguity",            "the top/bottom gadget"),
         ("PCP",  "CFG intersection empty",   "one grammar per row"),
         ("PCP",  "predicate-logic validity", "dominoes as axioms"),
         ("PCP",  "tiling the plane",         "Wang tiles")]

def reachable(edges, src):
    out, frontier = {src}, {src}
    while frontier:
        nxt = {b for a, b, _ in edges if a in frontier} - out
        out |= nxt; frontier = nxt
    return out

### Rice's theorem, as a predicate over properties

In [ ]:
def is_nontrivial_language_property(prop, examples):
    # prop maps a LANGUAGE (a set of strings) to True/False
    vals = {prop(L) for L in examples}
    return len(vals) > 1

## 3. Tests

Everything is reachable from $A_{TM}$.

In [ ]:
r = reachable(EDGES, "A_TM")
print("reachable from A_TM :")
for x in sorted(r): print("   ", x)
assert len(r) == 8

The chains, spelled out.

In [ ]:
def paths(edges, src, dst, acc=()):
    if src == dst: return [acc + (src,)]
    out = []
    for a, b, why in edges:
        if a == src and b not in acc:
            out += paths(edges, b, dst, acc + (src,))
    return out
for target in ["CFG ambiguity", "tiling the plane", "Halt_TM"]:
    for p in paths(EDGES, "A_TM", target):
        print("  " + "  ->  ".join(p))

Each edge is a computable $f$, and the edges compose.

In [ ]:
for a, b, why in EDGES:
    print("  %-22s -> %-26s  %s" % (a, b, why))
print("\n<=m is transitive, so a chain of edges IS a reduction.")

**Rice's theorem** covers the whole first branch at once.

In [ ]:
LANGS = [set(), {'0'}, {'0', '1'}, {'0' * k for k in range(5)}]
PROPS = {
 'is empty'        : lambda L: L == set(),
 'contains 0'      : lambda L: '0' in L,
 'is finite'       : lambda L: len(L) < 3,
 'TRIVIAL: True'   : lambda L: True,
 'TRIVIAL: False'  : lambda L: False,
}
for name, p in PROPS.items():
    nt = is_nontrivial_language_property(p, LANGS)
    print("  %-18s non-trivial? %-6s -> %s"
          % (name, nt, "UNDECIDABLE by Rice" if nt else "decidable (trivially)"))
assert is_nontrivial_language_property(PROPS['is empty'], LANGS)
assert not is_nontrivial_language_property(PROPS['TRIVIAL: True'], LANGS)

What Rice does **not** cover.

In [ ]:
NOT_RICE = [("does M have 7 states",       "a property of the MACHINE, not the language"),
            ("does M ever move left",      "a property of the machine's behaviour, not L(M)"),
            ("is this PCP instance solvable", "not about a machine at all")]
for a, b in NOT_RICE: print("  %-32s %s" % (a, b))
print("\nRice applies to properties of L(M).  The others need their own proofs --")
print("which is exactly why PCP is a useful separate source.")

The practical advice.

In [ ]:
print("When you meet a problem you suspect is undecidable:")
print("  1. is it a non-trivial property of L(M)?   -> Rice, done")
print("  2. is it a matching/combinatorial problem? -> try PCP")
print("  3. otherwise                               -> reduce from A_TM or Halt_TM")
print()
print("Diagonalize once.  Reduce forever.")

## 4. Exercises


1. State Rice's theorem precisely. Which two languages does the proof need?
2. Is "does $M$ halt in fewer than 100 steps?" decidable? Why does Rice not apply?
3. Add one more node to the reduction graph and justify the edge.

In [ ]:
# Your work for the exercises above.